In [1]:
###########################################################################
## Basic
###########################################################################
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("""<style>div.output_area{max-height:10000px;overflow:scroll;}</style>"""))


###########################################################################
## Warnings
###########################################################################
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning) 
warnings.filterwarnings("ignore", category=FutureWarning) 


###########################################################################
## Utils
###########################################################################
from timeUtils import timestat
from fileIO import fileIO

###########################################################################
## DB
###########################################################################
from masterManualEntries import masterManualEntries
from masterManualEntriesUtils import masterManualEntriesUtils
from masterDBGate import masterDBGate, masterDBIgnoreGate
from masterUtils import masterUtils

###########################################################################
## General
###########################################################################
from pandas import isna, notna, Series, DataFrame, concat
from uuid import uuid4

###########################################################################
## Sys
###########################################################################
import sys
print("Python: {0}".format(sys.version))

Python: 3.9.7 (default, Sep 16 2021, 08:50:36) 
[Clang 10.0.0 ]


In [2]:
io      = fileIO()
mu      = masterUtils()
#meu     = masterManualEntriesUtils()
mdbGate = masterDBGate()

In [63]:
from dbBase import dbBase
from artistIDBase import dbArtistID
from artistModValue import artistModValue
from masterArtistNameCorrection import masterArtistNameCorrection
from fsUtils import dirUtil,fileUtil

##################################################################################################################
# Base Class
##################################################################################################################
class dbIOBase:
    def __init__(self, db):
        self.db        = db
        self.disc      = dbBase(self.db.lower())
        self.aid       = dbArtistID(self.db)
        self.getpsid   = self.aid.getpsid
        self.getModVal = self.aid.getModVal
        self.manc      = masterArtistNameCorrection()
        
        
    ##############################################################################################################
    # Helper Functions
    ##############################################################################################################
    def getArtistPSID(self, artistName):
        return self.getpsid(self.manc.clean(artistName))
    
    def getModValDir(self, artistID):
        return dirUtil(self.disc.getArtistsDir()).join(str(self.getModVal(artistID)))
    
    
    ##############################################################################################################
    # Metadata Classes
    ##############################################################################################################
    def getBasicMetadata(self, artistData):
        return basicInfo(name=artistData.artist.name, url=artistData.url.url)
    
    def getGenreMetadata(self, artistData):
        return genreMetadata(None)
    
    def getLinkMetadata(self, artistData):
        return linkMetadata(archetype=None, groups=None, members=None, related=None)
    
    def getMetricMetadata(self, artistData):
        return linkMetadata(counts=None, quantile=None)
    
    def getBioMetadata(self, artistData):
        return linkMetadata(country=None, start=None, end=None)
    
    
##################################################################################################################
# Metadata Classes
##################################################################################################################
class basicMetadata:
    def __init__(self, name, url):
        self.name = name
        self.url  = url
        
class genreMetadata:
    def __init__(self, genres):
        if isinstance(genres,list):
            self.genres = genres
        elif genres is None:
            self.genres = []
        else:
            raise ValueError("Genre data of type {0} is not allowed".format(type(genres)))
            
class linkMetadata:
    def __init__(self, archetype, groups, members, related):
        self.archetype = archetype
        self.groups = groups
        self.members = members
        self.related = related
            
class metricMetadata:
    def __init__(self, counts, quantile):
        self.counts = counts
        self.quantile = quantile
            
class bioMetadata:
    def __init__(self, country, start, end):
        self.country = country
        self.start = start
        self.end = end


In [64]:
from artistDeezerAPI import artistDeezerAPI

class dbIODeezerAPI(dbIOBase):
    def __init__(self):
        super().__init__("DeezerAPI")
        self.artist = eval("artist{0}(self.disc)".format(self.db))
        self.io = fileIO()
        
        
    def getArtistSearchSavename(self, artistName):
        return fileUtil(self.getModValDir(psID)).join("{0}.p".format(self.getArtistPSID(artistName)))
    
    def saveArtistSearch(self, artistName, artistSearch):
        self.io.save(idata=artistSearch, ifile=self.getArtistSearchSavename(artistName))

In [65]:
from artistSpotify import artistSpotify
from fsUtils import fileUtil

class dbIOSpotify(dbIOBase):
    def __init__(self):
        super().__init__("Spotify")
        self.artist = eval("artist{0}(self.disc)".format(self.db))
        self.io = fileIO()
        
        
    ##############################################################################################################
    # File IO
    ##############################################################################################################
    def getArtistAlbumsSavename(self, artistID):
        return fileUtil(self.getModValDir(artistID)).join("{0}.p".format(artistID))
    
    def saveArtistAlbums(self, artistID, artistAlbums):
        self.io.save(idata=artistAlbums, ifile=self.getArtistAlbumsSavename(artistID))
    
    
    ##############################################################################################################
    # Metadata IO
    ##############################################################################################################
    def getGenreMetadata(self, artistData):
        return genreMetadata(artistData.profile.genres)
    
    def getLinkMetadata(self, artistData):
        return linkMetadata(archetype=artistData.profile.general.get("Type"), groups=None, members=None, related=None)
    
    def getMetricMetadata(self, artistData):
        return linkMetadata(archetype=artistData.profile.general.get("Type"), groups=None, members=None, related=None)
    
    def getBioMetadata(self, artistData):
        return linkMetadata(archetype=artistData.profile.general.get("Type"), groups=None, members=None, related=None)

In [ ]:
modValData = dbIO.disc.getDBModValData(0)

In [60]:
dbIO = dbIOSpotify()

In [53]:
modValData['5IH6FPUwQTxPSXurCrcIov'][1].profile.general

{'Type': 'artist'}

In [ ]:
genre = {}
relationship = {}
metric = {}
bio = {}

In [62]:
dbIO.getLinkMetadata(modValData['5IH6FPUwQTxPSXurCrcIov'][0]).__dict__

{'archetype': 'artist', 'groups': None, 'members': None, 'related': None}

In [38]:
modValData['5IH6FPUwQTxPSXurCrcIov'][1].show()

Artist Data Class
-------------------------
Artist:  Alec Benjamin
Meta:    None
         https://open.spotify.com/artist/5IH6FPUwQTxPSXurCrcIov
Info:    None
         None
         2022-01-25 11:52:56.374166
URL:     https://open.spotify.com/artist/5IH6FPUwQTxPSXurCrcIov
ID:      5IH6FPUwQTxPSXurCrcIov
Profile: {'general': {'Type': 'artist'}, 'genres': ['alt z', 'dream smp', 'electropop', 'pop'], 'tags': None, 'external': {'SpotifyAPI': {'URL': 'https://api.spotify.com/v1/artists/5IH6FPUwQTxPSXurCrcIov', 'URI': 'spotify:artist:5IH6FPUwQTxPSXurCrcIov'}}, 'extra': {'Followers': 3232459.0, 'Popularity': 80}, 'err': None}
Pages:   {'ppp': 1, 'tot': 1, 'pages': 1, 'err': None, 'more': False, 'redo': False}
Media:   {'counts': {'Releases': {'album + album': 2, 'single + single': 12, 'appears_on + album': 205}}, 'err': None}
   album + album
      These Two Windows
      Narrated For You
   single + single
      成长 (Older)
      Older
      Change My Clothes
      The Way You Felt (Acoustic 

In [28]:
dbIO.getGenreMetadata()

AttributeError: 'Series' object has no attribute 'profile'

In [5]:
from timeUtils import timestat
from fileIO import fileIO
from dbArtistsBase import dbArtistsBase
from pandas import Series

    

##################################################################################################################
# Collect Metadata About Artists
##################################################################################################################
class dbParseMetadata(dbArtistsBase):
    def __init__(self, dbIO, modVal=None):
        super().__init__(dbIO)
        self.dbIO = dbArtists
        self.io   = fileIO()
        
        _ = self.parse(modVal) if isinstance(modVal,int) else None
            
            
    def parse(self, modVal, **kwargs):
        self.modVal    = modVal
        self.dbData    = self.disc.getDBModValData(modVal)
        self.createArtistMetadata()
        self.createAlbumMetadata()
    
    
    def createArtistMetadata(self):
        ts = timestat("Creating Artist Name Metadata For ModVal={0}".format(self.modVal))
        basicMetadata = self.dbData.apply(lambda x: basicInfo(x.artist.name, x.url.url))
        
        artistIDMetadata = {str(artistID): [artistData.artist.name, artistData.url.url] for artistID,artistData in self.dbData.items() if artistData.artist.name is not None}
        artistIDMetadata = Series(artistIDMetadata)
        
        print("Saving [{0}] {1} Entries To {2}".format(len(artistIDMetadata), "ID => Name/URL", self.disc.getMetadataArtistFilename(self.modVal)))
        self.disc.saveMetadataArtistData(idata=artistIDMetadata, modVal=self.modVal)
        
        ts.stop()
        
        
    def createAlbumMetadata(self):
        ts = timestat("Creating Artist Album Metadata For ModVal={0}".format(self.modVal))        
        artistIDMetadata = {}
        errs = {}
        for artistID,artistData in self.dbData.items():
            artistID = str(artistID)
            if artistData.artist.name is None:
                continue
            artistIDMetadata[artistID] = {}
            for mediaName,mediaData in artistData.media.media.items():
                try:
                    albumURLs  = {mediaValues.code: mediaValues.url for mediaValues in mediaData}
                    albumNames = {mediaValues.code: mediaValues.album for mediaValues in mediaData}
                    artistIDMetadata[artistID][mediaName] = albumNames #, albumURLs]  
                except:
                    errs[artistID] = artistData.artist.name
                    #print(artistID,'\t',mediaName)
        artistIDMetadata = Series(artistIDMetadata)
        
        print("Saving [{0}] {1} Entries To {2}".format(len(artistIDMetadata), "ID => AlbumNames", self.disc.getMetadataAlbumFilename(self.modVal)))
        self.disc.saveMetadataAlbumData(idata=artistIDMetadata, modVal=self.modVal)    
        
        ts.stop()

In [6]:
modValData.apply(lambda x: basicInfo(x.artist.name, x.url.url))

5IH6FPUwQTxPSXurCrcIov    <__main__.basicInfo object at 0x7f7f60e61460>
49e4v89VmlDcFCMyDv9wQ9    <__main__.basicInfo object at 0x7f7f60e616a0>
2Waw2sSbqvAwK8NwACNjVo    <__main__.basicInfo object at 0x7f7f60e61b50>
4xFUf1FHVy696Q1JQZMTRj    <__main__.basicInfo object at 0x7f7f60e615e0>
4RVnAU35WRWra6OZ3CbbMA    <__main__.basicInfo object at 0x7f7f60e616d0>
                                              ...                      
73G0P8aYEnVht7jjliF7Ok    <__main__.basicInfo object at 0x7f7f5b2a02e0>
748MGeLsgxl6GVGuDvHbsY    <__main__.basicInfo object at 0x7f7f5b2a01f0>
7HK83pzwHsZqiGchCqtMuD    <__main__.basicInfo object at 0x7f7f5b2a01c0>
7eRc93g1wju97Og3KYVsEB    <__main__.basicInfo object at 0x7f7f5b2a0070>
7n0XJOct9yv45fCWY3t0UJ    <__main__.basicInfo object at 0x7f7f5b2a00d0>
Length: 5543, dtype: object

In [3]:
disc = mu.getDisc("Spotify")
modValData = disc.getDBModValData(0)

# Spotify

In [ ]:
def getSpotifyArtistType(profile):
    retval = profile.general.get('Type') if isinstance(profile.general, dict) else None
    return retval

def getSpotifyArtistGenre(profile):
    retval = profile.genres if isinstance(profile.genres, dict) else None
    return retval

def getSpotifyArtistMetric(profile):
    retval = profile.extra if isinstance(profile.extra, dict) else None
    return retval

artistType   = {}
artistGenres = {}
artistMetric = {}
disc = mu.getDisc("Spotify")
ts = timestat("Getting Spotify Data (~1 minute)")
for modVal in range(100):
    modValData = disc.getDBModValData(modVal)
    artistType.update({artistID: getSpotifyArtistType(artistData.profile) for artistID,artistData in modValData.iteritems()})
    artistGenres.update({artistID: getSpotifyArtistGenre(artistData.profile) for artistID,artistData in modValData.iteritems()})
    artistMetric.update({artistID: getSpotifyArtistMetric(artistData.profile) for artistID,artistData in modValData.iteritems()})
    if (modVal+1) % 10 == 0:
        ts.update(n=modVal+1,N=100)
ts.stop()

artistType   = Series(artistType)
artistGenres = Series(artistGenres)
artistMetric = Series(artistMetric)

In [ ]:
genre = {}
relationship = {}
metric = {}
bio = {}

# MusicBrainz

In [162]:
from pandas import Timestamp

def getMusicBrainzArtistType(profile):
    retval = profile.general.get('ArtistType') if isinstance(profile.general, dict) else None
    return retval

def getMusicBrainzArtistAliases(profile):
    retval = profile.general.get('Aliases') if isinstance(profile.general, dict) else None
    return retval

def getMusicBrainzArtistCountry(profile):
    retval = profile.general.get('County') if isinstance(profile.general, dict) else None
    return retval

def getMusicBrainzArtistActiveDates(profile):
    formed = profile.general.get('Formed') if isinstance(profile.general, dict) else None
    formed = formed.year if isinstance(formed, Timestamp) else None
    disbanded = profile.general.get('Disbanded') if isinstance(profile.general, dict) else None
    disbanded = disbanded.year if isinstance(disbanded, Timestamp) else None
    retval = [formed,disbanded]
    return retval


## Need to associate Genres, Tags
artistType        = {}
artistAliases     = {}
artistCountry     = {}
artistActiveDates = {}
disc = mu.getDisc("MusicBrainz")
ts = timestat("Getting MusicBrainz Data (~6 minutes)")
for modVal in range(100):
    modValData = disc.getDBModValData(modVal)
    artistType.update({artistID: getMusicBrainzArtistType(artistData.profile) for artistID,artistData in modValData.iteritems()})
    artistAliases.update({artistID: getMusicBrainzArtistAliases(artistData.profile) for artistID,artistData in modValData.iteritems()})
    artistCountry.update({artistID: getMusicBrainzArtistCountry(artistData.profile) for artistID,artistData in modValData.iteritems()})
    artistActiveDates.update({artistID: getMusicBrainzArtistActiveDates(artistData.profile) for artistID,artistData in modValData.iteritems()})
    break
    if (modVal+1) % 10 == 0:
        ts.update(n=modVal+1,N=100)
ts.stop()

artistType        = Series(artistType)
artistAliases     = Series(artistAliases)
artistCountry     = Series(artistCountry)
artistActiveDates = Series(artistActiveDates)

Starting Process [Getting MusicBrainz Data (~6 minutes)]    ==> Time Is 2022-01-31 09:58:24
Process [Getting MusicBrainz Data (~6 minutes)] Took 3.6 Seconds    ==> Time Is 2022-01-31 09:58:27


# Discogs

In [149]:
def getDiscogsArtistRealName(profile):
    retval = profile.general.get('RealName') if isinstance(profile.general, dict) else None
    return retval

def getDiscogsArtistAliases(profile):
    retval = profile.general.get('Aliases') if isinstance(profile.general, dict) else None
    return retval

def getDiscogsArtistGroups(profile):
    retval = profile.general.get('Groups') if isinstance(profile.general, dict) else None
    return retval

def getDiscogsArtistMembers(profile):
    retval = profile.general.get('Members') if isinstance(profile.general, dict) else None
    return retval

artistRealName   = {}
artistAliases    = {}
artistGroups     = {}
artistMembers    = {}
disc = mu.getDisc("Discogs")
ts = timestat("Getting Discogs Data (~30 minutes)")
for modVal in range(100):
    modValData = disc.getDBModValData(modVal)
    artistRealName.update({artistID: getDiscogsArtistRealName(artistData.profile) for artistID,artistData in modValData.iteritems()})
    artistAliases.update({artistID: getDiscogsArtistAliases(artistData.profile) for artistID,artistData in modValData.iteritems()})
    artistGroups.update({artistID: getDiscogsArtistGroups(artistData.profile) for artistID,artistData in modValData.iteritems()})
    artistMembers.update({artistID: getDiscogsArtistMembers(artistData.profile) for artistID,artistData in modValData.iteritems()})
    break
    if (modVal+1) % 10 == 0:
        ts.update(n=modVal+1,N=100)
ts.stop()

artistRealName = Series(artistRealName)
artistAliases  = Series(artistAliases)
artistGroups   = Series(artistGroups)
artistMembers  = Series(artistMembers)

Starting Process [Getting Discogs Data (~30 minutes)]    ==> Time Is 2022-01-31 09:51:40
Process [Getting Discogs Data (~30 minutes)] Took 12.5 Seconds    ==> Time Is 2022-01-31 09:51:53


# RateYourMusic

In [146]:
from artistDBBase import artistDBLinkClass,artistDBTextClass

def getRateYourMusicItem(item):
    retval = None
    if isinstance(item,artistDBLinkClass):
        retval = {item.text: mu.getid("RateYourMusic", item.title)}
    elif isinstance(item,artistDBTextClass):
        retval = item.text
    return retval

def getRateYourMusicArtistGenre(profile):
    artistIDData = profile.genres if isinstance(profile.genres, list) else None
    retval = [getRateYourMusicItem(item) for item in artistIDData] if isinstance(artistIDData,list) else None
    return retval

def getRateYourMusicArtistAlsoKnownAs(profile):
    artistIDData = profile.general.get("Also Known As") if isinstance(profile.general, dict) else None
    retval = [getRateYourMusicItem(item) for item in artistIDData] if isinstance(artistIDData,list) else None
    return retval

def getRateYourMusicArtistMembers(profile):
    artistIDData = profile.general.get("Members") if isinstance(profile.general, dict) else None
    retval = [getRateYourMusicItem(item) for item in artistIDData] if isinstance(artistIDData,list) else None
    return retval

def getRateYourMusicArtistMemberOf(profile):
    artistIDData = profile.extra.get("Member of") if isinstance(profile.extra, dict) else None
    retval = [getRateYourMusicItem(item) for item in artistIDData] if isinstance(artistIDData,list) else None
    return retval

def getRateYourMusicArtistMetric(profile):
    artistIDData = profile.external.get("Lists") if isinstance(profile.external, dict) else None
    retval = {"Lists": len(artistIDData)} if isinstance(artistIDData,list) else {"Lists": 0}
    return retval

def fixRateYourMusicArtistMembers(item):
    retval = None
    if item is None:
        pass
    elif isinstance(item,list):
        if all([isinstance(x,dict) for x in item]):
            retval = item
        elif all([isinstance(x,str) for x in item]) and len(item) == 1:
            retval = [y+")" for y in item[0].split("), ")]
        else:
            raise ValueError("Can't fix item [{0}]".format(item))
    else:
        raise ValueError("Can't fix item [{0}]".format(item))
        
    return retval

def fixRateYourMusicArtistAlsoKnownAs(item):
    retval = None
    if item is None:
        pass
    elif isinstance(item,list):
        if all([isinstance(x,dict) for x in item]):
            retval = item
        elif all([isinstance(x,str) for x in item]) and len(item) == 1:
            retval = [y for y in item[0].split(", ")]
        else:
            raise ValueError("Can't fix item [{0}]".format(item))
    else:
        raise ValueError("Can't fix item [{0}]".format(item))
        
    return retval

def fixRateYourMusicArtistEmptyIDs(item):
    retval = None
    if item is None:
        pass
    elif isinstance(item,list):
        if all([isinstance(x,dict) for x in item]):
            retval = getFlatList([x.keys() for x in item])
        else:
            raise ValueError("Can't fix item [{0}]".format(item))
    else:
        raise ValueError("Can't fix item [{0}]".format(item))
    return retval    

artistGenres      = {}
artistAlsoKnownAs = {}
artistMembers     = {}
artistMemberOf    = {}
artistMetric      = {}
disc = mu.getDisc("RateYourMusic")
ts = timestat("Getting RateYourMusic Data (~1 minute)")
for modVal in range(100):
    modValData = disc.getDBModValData(modVal)
    artistGenres.update({artistID: getRateYourMusicArtistGenre(artistData.profile) for artistID,artistData in modValData.iteritems()})
    artistAlsoKnownAs.update({artistID: getRateYourMusicArtistAlsoKnownAs(artistData.profile) for artistID,artistData in modValData.iteritems()})
    artistMembers.update({artistID: getRateYourMusicArtistMembers(artistData.profile) for artistID,artistData in modValData.iteritems()})
    artistMemberOf.update({artistID: getRateYourMusicArtistMemberOf(artistData.profile) for artistID,artistData in modValData.iteritems()})
    artistMetric.update({artistID: getRateYourMusicArtistMetric(artistData.profile) for artistID,artistData in modValData.iteritems()})
    if (modVal+1) % 10 == 0:
        ts.update(n=modVal+1,N=100)
ts.stop()

artistGenres      = Series(artistGenres)
artistGenres      = artistGenres.apply(fixRateYourMusicArtistEmptyIDs)
artistAlsoKnownAs = Series(artistAlsoKnownAs)
artistAlsoKnownAs = artistAlsoKnownAs.apply(fixRateYourMusicArtistAlsoKnownAs)
artistMembers     = Series(artistMembers)
artistMembers     = artistMembers.apply(fixRateYourMusicArtistMembers)
artistMemberOf    = Series(artistMemberOf)
artistMetric      = Series(artistMetric)

Starting Process [Getting RateYourMusic Data (~1 minute)]    ==> Time Is 2022-01-31 09:49:46
10/100     : Process [Getting RateYourMusic Data (~1 minute)] Has Run For 4.4 Seconds.  ETA is 39.6 Seconds
20/100     : Process [Getting RateYourMusic Data (~1 minute)] Has Run For 9.2 Seconds.  ETA is 36.8 Seconds
30/100     : Process [Getting RateYourMusic Data (~1 minute)] Has Run For 13.5 Seconds.  ETA is 31.5 Seconds
40/100     : Process [Getting RateYourMusic Data (~1 minute)] Has Run For 18.1 Seconds.  ETA is 27.2 Seconds
50/100     : Process [Getting RateYourMusic Data (~1 minute)] Has Run For 24.1 Seconds.  ETA is 24.1 Seconds
60/100     : Process [Getting RateYourMusic Data (~1 minute)] Has Run For 28.6 Seconds.  ETA is 19.1 Seconds
70/100     : Process [Getting RateYourMusic Data (~1 minute)] Has Run For 33.3 Seconds.  ETA is 14.3 Seconds
80/100     : Process [Getting RateYourMusic Data (~1 minute)] Has Run For 38.1 Seconds.  ETA is 9.5 Seconds
90/100     : Process [Getting RateYour

# AllMusic

In [106]:
from artistDBBase import artistDBLinkClass,artistDBTextClass
from listUtils import getFlatList

def getAllMusicItem(item):
    retval = None
    if isinstance(item,artistDBLinkClass):
        retval = {item.text: mu.getid("AllMusic", item.href)}
    elif isinstance(item,artistDBTextClass):
        retval = item.text
    return retval

def getAllMusicArtistGenre(profile):
    artistIDData = profile.genres if isinstance(profile.genres, list) else None
    retval = [getAllMusicItem(item) for item in artistIDData] if isinstance(artistIDData,list) else None
    return retval

def getAllMusicArtistTag(profile):
    artistIDData = profile.tags if isinstance(profile.tags, list) else None
    retval = [getAllMusicItem(item) for item in artistIDData] if isinstance(artistIDData,list) else None
    return retval

def getAllMusicArtistAliases(profile):
    artistIDData = profile.general.get("aliases") if isinstance(profile.general, dict) else None
    retval = [getAllMusicItem(item) for item in artistIDData] if isinstance(artistIDData,list) else None
    return retval

def getAllMusicArtistMembers(profile):
    artistIDData = profile.general.get("group-members") if isinstance(profile.general, dict) else None
    retval = [getAllMusicItem(item) for item in artistIDData] if isinstance(artistIDData,list) else None
    return retval

def getAllMusicArtistMemberOf(profile):
    artistIDData = profile.general.get("member-of") if isinstance(profile.general, dict) else None
    retval = [getAllMusicItem(item) for item in artistIDData] if isinstance(artistIDData,list) else None
    return retval

def getAllMusicArtistActiveDates(profile):
    artistIDData = profile.general.get("active-dates") if isinstance(profile.general, dict) else None
    retval = [getAllMusicItem(item) for item in artistIDData] if isinstance(artistIDData,list) else None
    return retval

def fixAllMusicArtistEmptyIDs(item):
    retval = None
    if item is None:
        pass
    elif isinstance(item,list):
        if all([isinstance(x,dict) for x in item]):
            retval = getFlatList([x.keys() for x in item])
        else:
            raise ValueError("Can't fix item [{0}]".format(item))
    else:
        raise ValueError("Can't fix item [{0}]".format(item))
    return retval    

def fixAllMusicArtistSplitText(item):
    retval = None
    if item is None:
        pass
    elif isinstance(item,list):
        if len(item) == 1 and isinstance(item[0],str):
            retval = item[0].split("\n")[-1].strip()
        elif all([isinstance(x,dict) for x in item]):
            retval = []
            for x in item:
                items = list(x.items())[0]
                key,value = items[0],items[1]
                key = [y for y in key.split("\n") if len(y) > 0]
                if len(key) == 1:
                    retval.append({key[0].strip(): value})
                else:
                    raise ValueError("Can't fix item [{0}]".format(item))
        else:
            raise ValueError("Can't fix item [{0}]".format(item))
    else:
        raise ValueError("Can't fix item [{0}]".format(item))
    return retval

artistGenres   = {}
artistTags     = {}
artistAliases  = {}
artistMembers  = {}
artistMemberOf = {}
artistActiveDates = {}
disc = mu.getDisc("AllMusic")
ts = timestat("Getting AllMusic Data (~3 1/2 minutes)")
for modVal in range(100):
    modValData = disc.getDBModValData(modVal)
    artistGenres.update({artistID: getAllMusicArtistGenre(artistData.profile) for artistID,artistData in modValData.iteritems()})
    artistTags.update({artistID: getAllMusicArtistTag(artistData.profile) for artistID,artistData in modValData.iteritems()})
    artistAliases.update({artistID: getAllMusicArtistAliases(artistData.profile) for artistID,artistData in modValData.iteritems()})
    artistMembers.update({artistID: getAllMusicArtistMembers(artistData.profile) for artistID,artistData in modValData.iteritems()})
    artistMemberOf.update({artistID: getAllMusicArtistMemberOf(artistData.profile) for artistID,artistData in modValData.iteritems()})
    artistActiveDates.update({artistID: getAllMusicArtistActiveDates(artistData.profile) for artistID,artistData in modValData.iteritems()})
    break
    if (modVal+1) % 10 == 0:
        ts.update(n=modVal+1,N=100)
ts.stop()

artistGenres      = Series(artistGenres)
artistGenres      = artistGenres.apply(fixAllMusicArtistEmptyIDs)
artistTags        = Series(artistTags)
artistTags        = artistTags.apply(fixAllMusicArtistEmptyIDs)
artistAliases     = Series(artistAliases)
artistAliases     = artistAliases.apply(fixAllMusicArtistSplitText)
artistMembers     = Series(artistMembers)
artistMembers     = artistMembers.apply(fixAllMusicArtistSplitText)
artistMemberOf    = Series(artistMemberOf)
artistMemberOf    = artistMemberOf.apply(fixAllMusicArtistSplitText)
artistActiveDates = Series(artistActiveDates)
artistActiveDates = artistActiveDates.apply(fixAllMusicArtistSplitText)

Starting Process [Getting AllMusic Data (~3 1/2 minutes)]    ==> Time Is 2022-01-31 09:35:08
Process [Getting AllMusic Data (~3 1/2 minutes)] Took 2.1 Seconds    ==> Time Is 2022-01-31 09:35:10


# Genius

In [ ]:
artistActiveDates = {}
disc = mu.getDisc("LastFMAPI")
ts = timestat("Getting LastFMAPI Data (~3 1/2 minutes)")
for modVal in range(100):
    modValData = disc.getDBModValData(modVal)
    break
    artistActiveDates.update({artistID: getAllMusicArtistActiveDates(artistData.profile) for artistID,artistData in modValData.iteritems()})
    break
    if (modVal+1) % 10 == 0:
        ts.update(n=modVal+1,N=100)
ts.stop()

#artistGenres      = Series(artistGenres)
#artistGenres      = artistGenres.apply(fixAllMusicArtistEmptyIDs)

# LastFM

In [164]:
artistActiveDates = {}
disc = mu.getDisc("LastFMAPI")
ts = timestat("Getting LastFMAPI Data (~3 1/2 minutes)")
for modVal in range(100):
    modValData = disc.getDBModValData(modVal)
    break
    artistActiveDates.update({artistID: getAllMusicArtistActiveDates(artistData.profile) for artistID,artistData in modValData.iteritems()})
    break
    if (modVal+1) % 10 == 0:
        ts.update(n=modVal+1,N=100)
ts.stop()

#artistGenres      = Series(artistGenres)
#artistGenres      = artistGenres.apply(fixAllMusicArtistEmptyIDs)

Starting Process [Getting LastFMAPI Data (~3 1/2 minutes)]    ==> Time Is 2022-01-31 10:01:50
Process [Getting LastFMAPI Data (~3 1/2 minutes)] Took 5.8 Seconds    ==> Time Is 2022-01-31 10:01:55


In [169]:
modValData['11290763000'].profile.get()

{'general': None,
 'genres': None,
 'tags': None,
 'external': {'MusicBrainzID': None},
 'extra': None,
 'err': None}

# AlbumOfTheYear

In [179]:
from artistDBBase import artistDBLinkClass,artistDBTextClass
from listUtils import getFlatList

def getAlbumOfTheYearItem(item):
    retval = None
    if isinstance(item,artistDBLinkClass):
        retval = {item.text: mu.getid("AlbumOfTheYear", item.href)}
    elif isinstance(item,artistDBTextClass):
        retval = item.text
    return retval

def getAlbumOfTheYearArtistGenre(profile):
    artistIDData = profile.genres if isinstance(profile.genres, list) else None
    retval = [getAllMusicItem(item) for item in artistIDData] if isinstance(artistIDData,list) else None
    return retval

def fixAlbumOfTheYearArtistEmptyIDs(item):
    retval = None
    if item is None:
        pass
    elif isinstance(item,list):
        if all([isinstance(x,dict) for x in item]):
            retval = getFlatList([x.keys() for x in item])
        else:
            raise ValueError("Can't fix item [{0}]".format(item))
    else:
        raise ValueError("Can't fix item [{0}]".format(item))
    return retval    

artistGenres = {}
disc = mu.getDisc("AlbumOfTheYear")
ts = timestat("Getting AlbumOfTheYear Data (~3 1/2 minutes)")
for modVal in range(100):
    modValData = disc.getDBModValData(modVal)
    modValData = modValData if isinstance(modValData, Series) else Series(modValData)
    artistGenres.update({artistID: getAlbumOfTheYearArtistGenre(artistData.profile) for artistID,artistData in modValData.iteritems()})
    break
    if (modVal+1) % 10 == 0:
        ts.update(n=modVal+1,N=100)
ts.stop()

artistGenres      = Series(artistGenres)
artistGenres      = artistGenres.apply(fixAlbumOfTheYearArtistEmptyIDs)

Starting Process [Getting AlbumOfTheYear Data (~3 1/2 minutes)]    ==> Time Is 2022-01-31 10:21:41
Process [Getting AlbumOfTheYear Data (~3 1/2 minutes)] Took 0.5 Seconds    ==> Time Is 2022-01-31 10:21:42


# Ideas

In [ ]:
genre = {}
relationship = {}
metric = {}
bio = {}

In [ ]:
albumStats = {}